En este archivo se realiza extracción de datos de Eurostat, calcula el PIB per cápita y prepara el terreno para el cruce con la European Social Survey (ESS).

In [14]:
pip install pandas numpy eurostat pyreadstat matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
# IMPORTACIÓN DE LIBRERÍAS
import pandas as pd
import numpy as np
import eurostat
import os  # Importación obligatoria al inicio para evitar NameError
import sys
import pyreadstat  # Necesaria para leer archivos .sav de la ESS más adelante
import matplotlib.pyplot as plt
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [16]:
# Vamos a descargar el PIB (nama_10_gdp) y la Población (demo_pjan)
print("Descargando datos de Eurostat (esto puede tardar unos segundos)...")

# PIB a precios corrientes
df_gdp_raw = eurostat.get_data_df('nama_10_gdp')
# Población a 1 de enero
df_pop_raw = eurostat.get_data_df('demo_pjan')

print("Datos descargados.")

Descargando datos de Eurostat (esto puede tardar unos segundos)...
Datos descargados.


In [29]:
# Descargamos las inversiones en educación, cultura y sanidad del dataset COFOG:
print("Descargando datos de Educación, Sanidad y Cultura...")
df_cofog_raw = eurostat.get_data_df('gov_10a_exp')
print("Datos descargados.")

Descargando datos de Educación, Sanidad y Cultura...
Datos descargados.


In [30]:
df_gdp_raw.head(4)

,freq,unit,na_item,geo,1975,1976,1977,1978,1979,1980,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,CLV05_MEUR,B1G,AL,NaN,NaN,NaN,NaN,NaN,NaN,...,8305.9,8592.7,8914.3,9149.4,8927.2,9540.7,10039.3,10536.5,10873.1,NaN
1,A,CLV05_MEUR,B1G,AT,NaN,NaN,NaN,NaN,NaN,NaN,...,258221.3,264412.6,271627.6,276482.8,259477.4,271146.4,288700.8,285720.1,282992.9,284179.8
2,A,CLV05_MEUR,B1G,BA,NaN,NaN,NaN,NaN,NaN,NaN,...,9650.5,9970.4,10347.6,10642.3,10350.5,11071.9,11532.7,11768.6,12110.0,NaN
3,A,CLV05_MEUR,B1G,BE,NaN,NaN,NaN,NaN,NaN,NaN,...,320447.4,325019.4,331266.1,339917.3,325628.4,344167.9,358932.8,365409.7,368794.1,372153.4


In [31]:
df_pop_raw.head(4)

,freq,unit,age,sex,geo,1960,1961,1962,1963,1964,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,NR,TOTAL,F,AD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,37388.0,NaN,NaN,NaN,NaN,NaN,NaN
1,A,NR,TOTAL,F,AL,NaN,NaN,NaN,NaN,NaN,...,1417141.0,1423050.0,1431715.0,1432833.0,1425342.0,1419759.0,1406532.0,1394864.0,NaN,1194597.0
2,A,NR,TOTAL,F,AM,NaN,NaN,NaN,NaN,NaN,...,1569535.0,1567380.0,1564533.0,1563538.0,1562689.0,1565144.0,NaN,1571757.0,1577675.0,NaN
3,A,NR,TOTAL,F,AT,3757167.0,3773097.0,3794130.0,3814191.0,3836415.0,...,4427918.0,4460424.0,4483749.0,4501742.0,4522292.0,4535712.0,4553444.0,4619957.0,4643918.0,4664333.0


In [32]:
df_cofog_raw.head(4) 

,freq,unit,sector,cofog99,na_item,geo\TIME_PERIOD,1990,1991,1992,1993,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,MIO_EUR,S13,GF01,D1,AT,NaN,NaN,NaN,NaN,...,5694.8,5766.6,5968.1,6179.8,6401.1,6643.6,6936.9,7464.2,8270.6,NaN
1,A,MIO_EUR,S13,GF01,D1,BE,NaN,NaN,NaN,NaN,...,8854.5,9001.1,9092.2,9423.1,9890.6,10317.9,10928.2,11959.7,12586.5,NaN
2,A,MIO_EUR,S13,GF01,D1,BG,NaN,NaN,NaN,NaN,...,492.5,518.6,572.8,691.6,690.7,860.4,880.5,1048.8,1240.2,NaN
3,A,MIO_EUR,S13,GF01,D1,CH,NaN,NaN,NaN,NaN,...,9383.2,9373.4,9223.1,9656.0,10393.1,10489.6,11544.8,12465.5,13008.6,NaN


In [34]:
# LIMPIEZA Y FILTRADO INICIAL
# Eurostat usa nombres de columnas compuestos (ej. 'geo\\time'). 
# Esta función limpia los nombres para quedarnos solo con la parte importante.
def clean_columns(df):
    # Dividimos por la barra invertida y nos quedamos con la primera parte (ej. 'geo')
    df.columns = [str(col).split('\\')[0] for col in df.columns]
    # Por si acaso el año viene como número, lo pasamos a string para evitar errores
    df.columns = [str(col) for col in df.columns]
    return df

df_gdp = clean_columns(df_gdp_raw)
df_pop = clean_columns(df_pop_raw)
df_inversiones = clean_columns(df_cofog_raw)

In [36]:
display(df_gdp.head(3))
display(df_pop.head(3))
display(df_inversiones.head(3))

,freq,unit,na_item,geo,1975,1976,1977,1978,1979,1980,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,CLV05_MEUR,B1G,AL,NaN,NaN,NaN,NaN,NaN,NaN,...,8305.9,8592.7,8914.3,9149.4,8927.2,9540.7,10039.3,10536.5,10873.1,NaN
1,A,CLV05_MEUR,B1G,AT,NaN,NaN,NaN,NaN,NaN,NaN,...,258221.3,264412.6,271627.6,276482.8,259477.4,271146.4,288700.8,285720.1,282992.9,284179.8
2,A,CLV05_MEUR,B1G,BA,NaN,NaN,NaN,NaN,NaN,NaN,...,9650.5,9970.4,10347.6,10642.3,10350.5,11071.9,11532.7,11768.6,12110.0,NaN


,freq,unit,age,sex,geo,1960,1961,1962,1963,1964,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,NR,TOTAL,F,AD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,37388.0,NaN,NaN,NaN,NaN,NaN,NaN
1,A,NR,TOTAL,F,AL,NaN,NaN,NaN,NaN,NaN,...,1417141.0,1423050.0,1431715.0,1432833.0,1425342.0,1419759.0,1406532.0,1394864.0,NaN,1194597.0
2,A,NR,TOTAL,F,AM,NaN,NaN,NaN,NaN,NaN,...,1569535.0,1567380.0,1564533.0,1563538.0,1562689.0,1565144.0,NaN,1571757.0,1577675.0,NaN


,freq,unit,sector,cofog99,na_item,geo,1990,1991,1992,1993,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,MIO_EUR,S13,GF01,D1,AT,NaN,NaN,NaN,NaN,...,5694.8,5766.6,5968.1,6179.8,6401.1,6643.6,6936.9,7464.2,8270.6,NaN
1,A,MIO_EUR,S13,GF01,D1,BE,NaN,NaN,NaN,NaN,...,8854.5,9001.1,9092.2,9423.1,9890.6,10317.9,10928.2,11959.7,12586.5,NaN
2,A,MIO_EUR,S13,GF01,D1,BG,NaN,NaN,NaN,NaN,...,492.5,518.6,572.8,691.6,690.7,860.4,880.5,1048.8,1240.2,NaN


In [40]:
# Elegimos el año más reciente con datos consolidados
ANIO = '2022'

# Filtramos PIB: Precios corrientes (CP_MEUR) e indicador PIB (B1GQ)
# Añadimos drop_duplicates para evitar que versiones con distinto ajuste estacional dupliquen filas
pib_data = df_gdp[(df_gdp['unit'] == 'CP_MEUR') & (df_gdp['na_item'] == 'B1GQ')][['geo', ANIO]]
pib_data = pib_data.drop_duplicates(subset=['geo']).rename(columns={ANIO: 'pib_millones'})

# Filtramos Población: Solo el total (age TOTAL) y total de sexos (sex T) para evitar duplicados
pop_data = df_pop[(df_pop['age'] == 'TOTAL') & (df_pop['sex'] == 'T')][['geo', ANIO]]
pop_2022 = pop_data.drop_duplicates(subset=['geo']).rename(columns={ANIO: 'poblacion'})

# --- FILTRADO DE EDUCACIÓN, SANIDAD Y CULTURA ---
# Seleccionamos % del PIB y Gasto Total
inv_base = df_inversiones[(df_inversiones['unit'] == 'MIO_EUR')].copy()
# Salud (GF07)
salud_data = inv_base[inv_base['cofog99'] == 'GF07'][['geo', ANIO]].drop_duplicates(subset=['geo']).rename(columns={ANIO: 'salud_mio'})
# Cultura (GF08)
cultura_data = inv_base[inv_base['cofog99'] == 'GF08'][['geo', ANIO]].drop_duplicates(subset=['geo']).rename(columns={ANIO: 'cultura_mio'})
# Educación (GF09)
educacion_data = inv_base[inv_base['cofog99'] == 'GF09'][['geo', ANIO]].drop_duplicates(subset=['geo']).rename(columns={ANIO: 'educacion_mio'})


In [45]:
# CÁLCULO DE MEDIDA PROPIA: PIB PER CÁPITA
# Unimos PIB y población por 'geo':
df_final = pd.merge(pib_data, pop_2022, on='geo').dropna()
# Unimos las variables de Sanidad, Cultura y Educación:
# Usamos how='left' para asegurar que no perdemos países si falta algún dato social
df_final = pd.merge(df_final, salud_data, on='geo', how='left')
df_final = pd.merge(df_final, cultura_data, on='geo', how='left')
df_final = pd.merge(df_final, educacion_data, on='geo', how='left')

# Calculamos: (PIB en millones * 1.000.000) / Población
df_final['pib_per_capita'] = (df_final['pib_millones'].astype(float) * 1000000) / df_final['poblacion'].astype(float)

# Definimos los países que pertenecen a la Unión Europea (códigos ISO-2)
paises_ue = [
    'AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 
    'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 
    'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE'
]

# Filtramos solo UE y obtenemos el ranking:
paises_ordenados_por_pib = df_final[df_final['geo'].isin(paises_ue)].sort_values('pib_per_capita', ascending=False)
top_paises = df_final[df_final['geo'].isin(paises_ue)].sort_values('pib_per_capita', ascending=False).head(15)
print("\n--- RANKING UE: PIB E INVERSIÓN SOCIAL (Limpio) ---")
# Ahora mostramos también las nuevas columnas en el print final
display(top_paises[['geo', 'pib_per_capita', 'salud_mio', 'cultura_mio', 'educacion_mio']].reset_index(drop=True))


--- RANKING UE: PIB E INVERSIÓN SOCIAL (Limpio) ---


,geo,pib_per_capita,salud_mio,cultura_mio,educacion_mio
0,LU,118889.923566,967.0,284.8,2539.6
1,IE,101026.467922,10238.1,491.7,8923.4
2,DK,64794.855468,15320.3,2046.7,10704.3
3,NL,56496.988859,2447.0,2790.0,28857.0
4,SE,52351.065208,14812.8,1836.7,16699.9
5,AT,50048.530287,11470.9,1466.3,13487.3
6,BE,48347.084425,1859.5,2325.5,28251.1
7,FI,47967.454910,8128.0,1097.0,7491.0
8,DE,47928.013467,15957.0,12348.0,97636.0
9,FR,38976.807497,64323.6,13437.7,93150.7


In [46]:
# PREPARACIÓN PARA LA ESS
# Aquí guardamos la lista de nuestros 10 países objetivo
lista_paises_objetivo = top_paises['geo'].tolist()

# NOTA PARA EL FUTURO:
# Al descargar los archivos ESS9, ESS10 y ESS11:
# 1. Carga el archivo: ess_data, meta = pyreadstat.read_spss('archivo.sav')
# 2. Filtra: ess_top = ess_data[ess_data['cntry'].isin(lista_paises_objetivo)]
# 3. Recuerda usar ESS10SC para Alemania (DE) y Austria (AT) si no están en la R10 normal.

print("\nListo para el siguiente paso: Carga de microdatos de la ESS.")


Listo para el siguiente paso: Carga de microdatos de la ESS.


In [47]:
lista_paises_objetivo

['LU',
 'IE',
 'DK',
 'NL',
 'SE',
 'AT',
 'BE',
 'FI',
 'DE',
 'FR',
 'MT',
 'IT',
 'CY',
 'ES',
 'CZ']

In [ ]:
archivo_ess = 'ess_cumulative.sav'